# 🐦 RWKV: Receptance Weighted Key Values
## Versión alumno · Aprende la alternativa "Transformer + RNN"

En este notebook vas a:

1. 📖 **Entender** qué es RWKV y cómo combina Transformer + RNN
2. 🔍 **Descubrir** cómo logra paralelismo en entrenamiento e inferencia eficiente
3. ⚡ **Comparar** con Transformer y Mamba
4. ✏️ **Experimentar** con los tres modelos

---

### 💡 La idea brillante

RWKV = **Transformer entrenable** + **RNN eficiente**

La "magia": una atención **lineal** que puede reescribirse como **recurrencia**.

> 💬 Imagina un libro que puedes leer en paralelo (como un Transformer)
> pero para resumirlo, solo necesitas la última página (como un RNN)...

## 1. Preparación

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"📱 Dispositivo: {device}")

## 2. El dataset: Don Quijote

In [ ]:
data_path = Path("el_quijote.txt")

if not data_path.exists():
    print("📥 Descargando Don Quijote...")
    url = "https://www.gutenberg.org/files/2000/2000-0.txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text = response.text
    start_marker = "*** START OF THE PROJECT GUTENBERG"
    end_marker = "*** END OF THE PROJECT GUTENBERG"
    start_idx = text.find(start_marker) + len(start_marker)
    end_idx = text.find(end_marker)
    text = text[start_idx:end_idx].strip()
    data_path.write_text(text)
    print("✅ Descarga completada")

text = data_path.read_text()
print(f"📚 El Quijote: {len(text):,} caracteres")

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
print(f"🔤 Vocabulario: {vocab_size} caracteres")

def get_batch(batch_size=64, block_size=128, device='cpu'):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

---

## 3. El problema que resuelve RWKV

### 3.1 Los dos mundos

```
TRANSFORMER          vs           RNN (LSTM/GRU)
──────────────────────────────────────────────────
Paralelo ✓            →          Secuencial ✗
Lento en inferencia   →          Rápido en inferencia ✓
Memoria O(n²)         →          Memoria O(1) ✓
Calidad alta          →          Calidad limitada
```

### 3.2 ¿Y si pudiéramos tener lo mejor de ambos?

RWKV propone:
- **Entrenar** como Transformer (paralelo)
- **Inferir** como RNN (O(1) por token)

La clave: usar **atención lineal** que puede reescribirse como recurrencia.

### 🤔 Analogía: El examen vs el resumen

**Transformer** (como un examen):
```
Para responder pregunta 5, necesito consultar
todas las preguntas anteriores.
→ Cada respuesta requiere releer todo.
```

**RNN** (como un resumen progresivo):
```
Mantengo un "resumen" actualizado en mi cabeza.
Cada nueva información actualiza el resumen.
→ Solo necesito el resumen, no todo el texto.
```

**RWKV** (el truco):
```
Puedo "escribir" las respuestas en paralelo (entrenamiento)
Pero la "información" se累计 como en un RNN (inferencia)
```

## 4. Cómo funciona RWKV

### 4.1 Las componentes

RWKV usa 4 vectores por token:

| Componente | Significado | Analogía |
|------------|-------------|----------|
| **R**eceptance | Cuánto "acepta" el token | Filtro de entrada |
| **K**ey | La "clave" para matching | Atención Keys |
| **V**alue | El "valor" a propagar | Atención Values |
| **W**eight | Decay temporal | Cuánto "olvida" |

### 4.2 La recurrencia

```
Estado en t = Estado en t-1 × decay + r_t × k_t × v_t

Esto es O(1) por token → inferencia rápida
```

### 4.3 El time shift

Antes de procesar, RWKV "desliza" el input:

```
Input original:  [x_1, x_2, x_3, x_4, x_5]
Después shift:  [x_0, x_1, x_2, x_3, x_4]  (x_0 = padding)
```

Esto permite que cada posición "mire" un poco atrás.

In [ ]:
# ======================================
# ILUSTRACIÓN: Time Shift
# ======================================

print("📖 ILUSTRACIÓN DEL TIME SHIFT EN RWKV")
print("=" * 50)
print("\nSin time shift (posiciones獨立):")
tokens = ["E", "n", " ", "u", "n", " ", "l", "u", "g", "a", "r"]
print(f"   Pos:     {list(range(len(tokens)))}")
print(f"   Tokens:  {tokens}")

print("\nCon time shift (cada pos "mira atrás"):")
shifted = ["_"] + tokens[:-1]
print(f"   Pos:     {list(range(len(shifted)))}")
print(f"   Tokens:  {shifted}")

print("\n💡 El time shift da "memoria" sin recurrencia explícita")
print("   - La posición 5 "ve" parcialmente la posición 4")
print("   - Pero se calcula en paralelo (no secuencial)")

## 5. Los bloques de RWKV

In [ ]:
# ======================================
# 5.1 RMSNorm
# ======================================

class RMSNorm(nn.Module):
    """
    Root Mean Square Normalization
    """
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    
    def forward(self, x):
        output = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * output

In [ ]:
# ======================================
# 5.2 RWKV Time Mixing
# ======================================

class RWKVTimeMix(nn.Module):
    """
    RWKV Time Mixing: la "atención" de RWKV
    
    Usa:
    - receptance (r): puerta de entrada
    - key (k): para matching
    - value (v): información a propagar
    - time_decay (w): cuánto "olvida" en cada paso
    """
    
    def __init__(self, d_model, layer_id):
        super().__init__()
        self.d_model = d_model
        self.layer_id = layer_id
        
        # Parámetros de tiempo
        self.time_decay = nn.Parameter(torch.zeros(d_model))
        self.time_first = nn.Parameter(torch.zeros(d_model))
        
        # Proyecciones
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.output = nn.Linear(d_model, d_model, bias=False)
        
        # Time shift
        self.time_shift = nn.ZeroPad2d((0, 0, 1, -1))
    
    def forward(self, x, state=None):
        if state is None:
            state = {
                'w': torch.zeros_like(self.time_decay),
                'k': torch.zeros_like(x),
                'v': torch.zeros_like(x),
                'o': torch.zeros_like(x),
            }
        
        b, l, d = x.shape
        
        # Time shift
        x = self.time_shift(x)
        
        # Calcular r, k, v
        r = torch.sigmoid(self.receptance(x))
        k = self.key(x)
        v = self.value(x)
        
        # Decay exponencial
        w = torch.exp(self.time_decay)
        
        # Acumular con decay
        wkv = state['w'] * w + k * v
        rwkv = r * self.output(wkv / (state['w'] + k))
        
        # Nuevo estado
        new_state = {
            'w': state['w'] * w + k,
            'k': state['k'] * w + k,
            'v': state['v'] * w + k * v,
            'o': state['o'] * w + rwkv,
        }
        
        return x + rwkv, new_state

In [ ]:
# ======================================
# 5.3 RWKV Channel Mixing
# ======================================

class RWKVChannelMix(nn.Module):
    """
    RWKV Channel Mixing: el "FFN" de RWKV
    
    Mezcla información entre canales, como un FFN
    pero con receptance y time shift.
    """
    
    def __init__(self, d_model, d_ffn, layer_id):
        super().__init__()
        self.key = nn.Linear(d_model, d_ffn, bias=False)
        self.value = nn.Linear(d_ffn, d_model, bias=False)
        self.receptance = nn.Linear(d_model, d_model, bias=False)
        self.time_shift = nn.ZeroPad2d((0, 0, 1, -1))
    
    def forward(self, x):
        x = self.time_shift(x)
        r = torch.sigmoid(self.receptance(x))
        k = self.key(x)
        k = torch.relu(k).square()
        return r * self.value(k)

In [ ]:
# ======================================
# 5.4 RWKV Block completo
# ======================================

class RWKVBlock(nn.Module):
    """
    Bloque RWKV completo:
    1. Time Mixing (atención)
    2. Channel Mixing (FFN)
    """
    def __init__(self, d_model, layer_id=0, mix_attn=False):
        super().__init__()
        self.layer_id = layer_id
        self.d_model = d_model
        
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        
        if mix_attn:
            self.att = RWKVTimeMix(d_model, layer_id)
        else:
            self.att = nn.Identity()
        
        self.ffn = RWKVChannelMix(d_model, 4 * d_model, layer_id)
    
    def forward(self, x, state=None):
        if self.att is not None and not isinstance(self.att, nn.Identity):
            x, state = self.att(self.norm1(x), state)
        x = x + self.ffn(self.norm2(x))
        return x, state

In [ ]:
# ======================================
# 5.5 Attention Block (referencia)
# ======================================

class AttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
    
    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm(x + attn_out)
        x = self.norm(x + self.ff(x))
        return x

## 6. Los tres modelos

In [ ]:
# ======================================
# MODELOS COMPLETOS
# ======================================

class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=6):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, 512, d_model) * 0.02)
        self.layers = nn.ModuleList([
            AttentionBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        b, l = x.shape
        x = self.embed(x) + self.pos_embed[:, :l, :]
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))


class MiniRWKV(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_layers=12):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            RWKVBlock(d_model, layer_id=i, mix_attn=(i % 2 == 0)) 
            for i in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x, state=None):
        x = self.embed(x)
        new_state = []
        for layer in self.layers:
            x, layer_state = layer(x, state)
            new_state.append(layer_state)
        return self.head(self.norm(x)), new_state if state is not None else None


class HybridRWKV(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_layers=12):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        layers = []
        for i in range(n_layers):
            if i % 3 == 0:
                layers.append(AttentionBlock(d_model, n_heads=4))
            else:
                layers.append(RWKVBlock(d_model, layer_id=i, mix_attn=True))
        self.layers = nn.ModuleList(layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            if isinstance(layer, AttentionBlock):
                x = layer(x)
            else:
                x, _ = layer(x)
        return self.head(self.norm(x))

### 📊 Tabla comparativa

| Aspecto | Transformer | RWKV | Mamba |
|---------|-------------|------|-------|
| Entrenamiento | Paralelo ✓ | Paralelo ✓ | Parcial |
| Inferencia | O(n) | **O(1)** ✓ | O(n) |
| Memoria | O(n²) | O(n) | O(n) |
| Selección | Atención | Decay | SSM scan |
| Estado | No | Sí | Sí |

## 7. Entrenamiento y generación

In [ ]:
# ======================================
# FUNCIÓN DE ENTRENAMIENTO
# ======================================

def train_and_generate(model, name, epochs=8, batch_size=64, block_size=128, 
                       lr=4e-4, device='cuda', temperature=0.8, save_dir="models_rwkv"):
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    print(f"\n{'='*50}")
    print(f"🚀 ENTRENANDO: {name}")
    print(f"{'='*50}")
    
    losses = []
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        steps = 80
        
        for _ in tqdm(range(steps), desc=f"  Época {epoch+1}/{epochs}"):
            x, y = get_batch(batch_size, block_size, device)
            
            if isinstance(model, MiniRWKV):
                logits, _ = model(x, state=None)
            else:
                logits = model(x)
            
            loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        
        scheduler.step()
        avg_loss = total_loss / steps
        losses.append(avg_loss)
        
        print(f"  Época {epoch+1:2d} | Loss: {avg_loss:.4f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), f"{save_dir}/{name.replace(' ', '_')}_best.pt")
            print(f"    ⭐ Mejor modelo (loss: {best_loss:.4f})")
    
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, epochs+1), losses, marker='o', linewidth=2)
    plt.title(f"📉 Curva de pérdida - {name}")
    plt.xlabel("Época")
    plt.ylabel("Cross-Entropy Loss")
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"\n📝 Generando texto (temperatura={temperature})")
    print("-" * 60)
    model.eval()
    with torch.no_grad():
        context = torch.tensor([stoi[' ']], dtype=torch.long, device=device).unsqueeze(0)
        generated = []
        for _ in range(400):
            if isinstance(model, MiniRWKV):
                logits, _ = model(context, state=None)
            else:
                logits = model(context)
            
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            context = torch.cat([context, next_idx], dim=1)
            generated.append(itos[next_idx.item()])
        
        print(''.join(generated))
        print("\n" + "-" * 60)

## 8. 🏃 EJERCICIO FINAL: Entrena y compara

In [ ]:
# ======================================
# ENTRENAR LOS TRES MODELOS
# ======================================

if __name__ == "__main__":
    d_model = 128
    save_dir = "saved_models_rwkv"
    EPOCHS = 8
    TEMPERATURE = 0.85
    
    print(f"📱 Dispositivo: {device}")
    print(f"🔤 Vocabulario: {vocab_size} caracteres")
    
    models = [
        ("MiniTransformer", MiniTransformer(vocab_size, d_model, n_heads=4, n_layers=6)),
        ("Pure RWKV", MiniRWKV(vocab_size, d_model, n_layers=12)),
        ("Hybrid RWKV", HybridRWKV(vocab_size, d_model, n_layers=12))
    ]
    
    for name, model in models:
        train_and_generate(
            model, 
            name, 
            epochs=EPOCHS, 
            device=device, 
            temperature=TEMPERATURE,
            save_dir=save_dir
        )

## 9. 📝 Reflexión

### Pregunta 1: Eficiencia
RWKV tiene inferencia O(1). ¿Por qué importa esto?
- ¿Cuántas operaciones ahorras en 1000 tokens?

### Pregunta 2: Estado
RWKV mantiene estado. ¿Qué significa esto para:
- Generación muy larga?
- Agentes con memoria persistente?

### Pregunta 3: Comparación
Compara con Mamba:
- Mamba usa SSM scan
- RWKV usa linear attention + decay
- ¿Cuál prefieres para cada caso?

## 10. 🔬 Experimentos opcionales

### A. Time decay
```python# Cambia time_decay en RWKVTimeMix
# Decay grande → más "memoria"
# Decay pequeño → más "olvido"
```

### B. Frecuencia de mezcla
```python# Cambia i % 2 == 0 a i % 3 == 0
# Menos attention layers → más eficiencia
```

### C. RWKV vs Mamba
```python# Ambos tienen estado, ¿cuál converge más rápido?
```

---

## 📚 Resumen de lo que has aprendido

✅ **RWKV = Linear Attention + Recurrence**

✅ **Entrenamiento paralelo** como Transformer

✅ **Inferencia O(1)** como RNN

✅ **Receptance, Key, Value, Decay** son las componentes

✅ **Time shift** permite "memoria" sin recurrencia

---

> 💬 **Frase para recordar**:
> 
> *"RWKV: entrena como transformer, infiere como RNN."*

---

🎉 **¡Completado!**